In [1]:
import torch
import clip
import os
from torchvision import transforms, datasets
from tqdm import tqdm
import numpy as np

from utils import clip_analysis_utils
from utils import similarity

In [2]:
clip_name = "ALIGN" #"ViT-B/16"
# concept_set = "data/concept_sets/cub_filtered.txt"
dataset = "cub"
d_train = dataset + "_train"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

dataset_path = os.path.join(os.getcwd(), "Data\CUB_200_2011")
cub_concept_format = False  #wether the concept_set is in the original CUB format.
concept_file = "backup_final_concepts.txt"           #"run_final_concepts.txt" "cub_filtered.txt" "attributes_cub"
comparison_to_ground_truth = 'class_ground_truth' #which field to use for comparison if using ground truth concepts. class_ground_truth = probability per class straight from file. ground_truth_mean = mean over images in class from is_present attribute.
use_certainty = False  #whether to use certainty information from attributes file or just standard is_present

In [3]:
#definitions:
def my_similarity(a, b):
    return a @ b.T  #dot product

In [4]:
attributes_cub = clip_analysis_utils.parse_attributes_file(dataset_path, "attributes\\image_attribute_labels.txt", use_certainty)

In [5]:
#cleaned_concepts, cleaned_not_concepts, grouped_concepts = clip_analysis_utils.get_cleaned_concepts(dataset_path, "attributes.txt", cub_concept_format) #original CUB attributes
#cleaned_concepts, cleaned_not_concepts, grouped_concepts = clip_analysis_utils.get_cleaned_concepts(os.path.join(os.getcwd(), "Data\Concept_sets"), "cub_filtered.txt", cub_concept_format)  #filtered concepts
#cleaned_concepts, cleaned_not_concepts, grouped_concepts = clip_analysis_utils.get_cleaned_concepts(os.path.join(os.getcwd(), ""), "run_final_concepts.txt", cub_concept_format)  #filtered concepts
cleaned_concepts, cleaned_not_concepts, grouped_concepts = clip_analysis_utils.get_cleaned_concepts(os.path.join(os.getcwd(), "test"), concept_file, cub_concept_format)  #filtered concepts

In [6]:
mult_scores, not_mult_scores = clip_analysis_utils.get_clip_similarity_all(clip_name, cleaned_concepts, my_similarity, cleaned_not_concepts, device)

100%|██████████| 1/1 [00:00<00:00, 66.10it/s]


In [8]:
if len(cleaned_concepts) < len(attributes_cub[1])-1:
    print("Warning: fewer concepts loaded than in attributes file!")
    all_cleaned_concepts, _, _ = clip_analysis_utils.get_cleaned_concepts(dataset_path, "attributes.txt", True) #original CUB attributes
    print("All concepts:", len(all_cleaned_concepts))
    map_to_true_conceptIdx = [all_cleaned_concepts.index(item) for item in cleaned_concepts]
else:
    map_to_true_conceptIdx = list(range(len(cleaned_concepts)))


All concepts: 312


In [9]:
max_score = mult_scores.max().item()
min_score = mult_scores.min().item()
print("Max score:", max_score, "Min score:", min_score)

auc_results = np.zeros(len(cleaned_concepts))
for i in tqdm(range(len(cleaned_concepts))):
    scores = mult_scores[:, i]
    ground_truth = [attributes_cub[j+1][map_to_true_conceptIdx[i]+1][0] for j in range(len(scores))]      #(len(images)), list of ground truth for each image (starting at idx 0)
    auc_results[i] = clip_analysis_utils.plot_roc_curve(ground_truth, scores, file_name=f"ROC_concept_{i}_{cleaned_concepts[i]}", concept_name=cleaned_concepts[i])

100%|██████████| 57/57 [00:05<00:00,  9.59it/s]


In [10]:
auc_results

array([0.85267418, 0.75483693, 0.61984037, 0.64666896, 0.61557985,
       0.61768986, 0.81715171, 0.83610157, 0.69058567, 0.73286554,
       0.56986628, 0.69017438, 0.43248024, 0.90744762, 0.66177581,
       0.77923054, 0.70756225, 0.60758817, 0.60973204, 0.87996045,
       0.77192052, 0.57553041, 0.54624603, 0.73120739, 0.53503147,
       0.52581085, 0.55553066, 0.8578588 , 0.7979817 , 0.6183972 ,
       0.75726554, 0.86496979, 0.69102837, 0.83773015, 0.64495246,
       0.63390623, 0.79518229, 0.61139121, 0.76992677, 0.90990807,
       0.68137404, 0.67032619, 0.60079721, 0.58202086, 0.70891189,
       0.62887275, 0.67488592, 0.59805391, 0.76521686, 0.73161147,
       0.73233517, 0.80073285, 0.73844076, 0.85374491, 0.57284807,
       0.57346332, 0.80500001])

In [11]:
# index for results below 0.5
#np.where(auc_results < 0.5)
print("mean", np.mean(auc_results))
print("std", np.std(auc_results))

mean 0.6978986763276894
std 0.10960548737461732


In [12]:
idx = np.where(auc_results < 0.5)
print("Concepts with AUC below 0.5:")
for i in idx[0]:
    print(f"Concept: {cleaned_concepts[i]}, AUC: {auc_results[i]}")

Concepts with AUC below 0.5:
Concept: has bill color rufous, AUC: 0.43248024276007657
